# Unity Lab — Phase 1 Feasibility Probe (v3 — proven path)

Reproduces on a fresh Colab VM, in one **Run all**, everything proven working on 2026-11-09:

1. **GATE 1** — pinned Unity 6000.3.10f1 LTS editor installs and runs on a CPU VM (md5-gated, reusable).
2. **GATE 2** — Unity Personal license **activated headlessly** via the editor's bundled licensing client (no ULF, no Hub CLI, no Drive upload).
3. **GATE 3** — project + scene created purely by script.
4. **GATE 4** — Linux player built (timed, peak RAM logged) + **WebGL stretch** build.

## Before you run
- Runtime > Change runtime type > **CPU** (builds are CPU jobs).
- The **GATE 2** cell has placeholders `UNAME` / `UPASS`: fill your Unity ID email + password **locally before running** (never paste them into chat).
- Then Runtime > Run all. **No Drive popup needed anymore.**
- Rules: run the license cell **once**; if you see code `132.182`, wait 24h before any retry. If the tab must stay open, the keep-alive cell helps (best-effort).


In [ ]:
import subprocess
# --- Cell 1: system tools + VM spec ---
r = subprocess.run(['bash', '-lc', 'apt-get -qq update >/dev/null 2>&1; apt-get install -y -qq aria2 pigz xz-utils time >/dev/null 2>&1; echo APT_EXIT=$?'], capture_output=True, text=True)
print('apt:', (r.stdout or '').strip(), (r.stderr or '').strip()[-500:])
assert 'APT_EXIT=0' in r.stdout, 'apt failed'
r = subprocess.run(['bash', '-lc', 'echo == VM spec ==; nproc; free -m | head -2; df -h /content | tail -1; head -1 /etc/os-release'])


In [ ]:
from IPython.display import Javascript, display
# --- Cell 2: keep-alive (best-effort: silent WebAudio + heartbeat keeps the tab throttling away) ---
display(Javascript('''
(function(){
  if (window.__unitylab_ka) { console.log('keepalive already running'); return; }
  window.__unitylab_ka = true;
  try {
    var ctx = new (window.AudioContext || window.webkitAudioContext)();
    var o = ctx.createOscillator(); var g = ctx.createGain();
    g.gain.value = 0.0; o.connect(g); g.connect(ctx.destination); o.start();
    if (ctx.state === 'suspended' && ctx.resume) { ctx.resume(); }
  } catch(e) { console.log('keepalive audio unavailable:', e); }
  setInterval(function(){ console.log('unity-lab keepalive ' + new Date().toISOString()); }, 60000);
  console.log('keepalive started');
})();
'''))
print('KEEPALIVE_DISPLAYED (still keep the tab open; Colab may disconnect otherwise)')


In [ ]:
import json
# --- Cell 3: pinned release + config (no floating versions downstream) ---
CFG = {
    'VERSION': '6000.3.10f1',
    'REV': 'e35f0c77bd8e',
    'CMD5': 'd575da705f9f8d9d6bc4bf70546eca04',
    'EDITOR_URL': 'https://download.unity3d.com/download_unity/e35f0c77bd8e/LinuxEditorInstaller/Unity.tar.xz',
    'WEBGL_URL': 'https://download.unity3d.com/download_unity/e35f0c77bd8e/LinuxEditorTargetInstaller/UnitySetup-WebGL-Support-for-Editor-6000.3.10f1.tar.xz',
}
json.dump(CFG, open('/content/probe_config.json', 'w'))
print('Pinned:', CFG['VERSION'], '| rev:', CFG['REV'])


In [ ]:
import subprocess, time, glob, os, json
# --- Cell 4: download + extract editor --- GATE 1 --- (md5-gated, reusable) ---
print('PROBE_VERSION=3 (proven path)')
cfg = json.load(open('/content/probe_config.json'))
t0 = time.time()
ubin = '/content/unity_bin.txt'
already = False
if os.path.exists(ubin):
    p = open(ubin).read().strip()
    if p and os.path.exists(p):
        miss = subprocess.run(['bash', '-lc', 'ldd "%s" | grep "not found" || true' % p], capture_output=True, text=True).stdout.strip()
        already = (not miss)
UNITY_BIN = None
if already:
    UNITY_BIN = p
    print('GATE 1 SKIP — editor already installed at:', p)
else:
    r = subprocess.run(['bash', '-lc',
        'set -e; mkdir -p /tmp/xdg && chmod 700 /tmp/xdg; printf "pcm.!default {\n  type null\n}\nctl.!default {\n  type null\n}\n" > /etc/asound.conf; '
        'cd /content; echo STEP_D_START $(date +%T); '
        'if [ -f /content/Unity.tar.xz ] && md5sum /content/Unity.tar.xz | grep -qi "%s"; then echo SKIP_DOWNLOAD; '
        'else aria2c -x8 -s8 --console-log-level=warn -d /content -o Unity.tar.xz "%s"; fi; '
        'md5sum /content/Unity.tar.xz | grep -qi "%s" || { echo MD5_MISMATCH; exit 42; }; echo STEP_D_VERIFIED; '
        'mkdir -p /content/ue; tar -xJf /content/Unity.tar.xz -C /content/ue; echo STEP_TAR_DONE; rm -f /content/Unity.tar.xz; echo STEP_ALL_DONE;'
        % (cfg['CMD5'], cfg['EDITOR_URL'], cfg['CMD5'])], capture_output=True, text=True)
    print('BASH_RETURN_CODE:', r.returncode)
    if r.returncode != 0:
        print('--- BASH STDOUT (tail) ---\n' + (r.stdout or '(empty)')[-3000:])
        print('--- BASH STDERR (tail) ---\n' + (r.stderr or '(empty)')[-3000:])
        print('--- /content listing ---\n' + subprocess.run(['bash', '-lc', 'ls -la /content; df -h /content | tail -1'], capture_output=True, text=True).stdout)
        raise AssertionError('download/extract FAILED')
    cands = glob.glob('/content/ue/*/Editor/Unity') + glob.glob('/content/ue/Editor/Unity')
    assert cands, 'could not locate Editor/Unity binary'
    UNITY_BIN = cands[0]
    open(ubin, 'w').write(UNITY_BIN)
    print('download+extract: %.1f min' % ((time.time() - t0) / 60))
print('GATE 1 PASS — binary:', UNITY_BIN)
subprocess.run(['bash', '-lc', '"%s" -version' % UNITY_BIN])


In [ ]:
import subprocess, os
# --- Cell 5: license ACTIVATION --- GATE 2 --- (proven one-shot recipe; SKIPS if already licensed) ---
# Fill these two placeholders locally BEFORE running (never paste them into chat):
UNAME = 'PASTE_YOUR_EMAIL_HERE'
UPASS = 'PASTE_YOUR_PASSWORD_HERE'
# Cooldown rule: if output shows code 132.182, wait 24h before any retry (retries extend the block).

UNITY = open('/content/unity_bin.txt').read().strip()
UCLIENT = UNITY.replace('/Editor/Unity', '') + '/Editor/Data/Resources/Licensing/Client/Unity.Licensing.Client'
assert os.path.exists(UCLIENT), 'licensing client missing: ' + UCLIENT

def smoke():
    r = subprocess.run([UNITY, '-batchmode', '-quit', '-nographics', '-logFile', '/content/lic_smoke.log'],
                       capture_output=True, text=True, timeout=900)
    liclog = open('/content/lic_smoke.log', errors='ignore').read() if os.path.exists('/content/lic_smoke.log') else ''
    BAD = [s for s in ['no valid Unity Editor license', 'was not found', 'entitlement'] if s.lower() in liclog.lower()]
    return r.returncode, BAD, liclog

skip = False
lic_xml = os.path.expanduser('~/.config/unity3d/Unity/licenses/UnityEntitlementLicense.xml')
if os.path.exists(lic_xml):
    rc0, bad0, log0 = smoke()
    if rc0 == 0:
        skip = True
        print('GATE 2 SKIP — license already active on this VM (smoke rc 0)')
    else:
        print('license xml present but smoke failed (rc %d, bad: %s) — reactivating' % (rc0, bad0))

if not skip:
    DEPS = 'libx11-6 libx11-xcb1 libxext6 libxrandr2 libxinerama1 libxcursor1 libxi6 libfontconfig1 libglib2.0-0 libnss3 libdbus-1-3 libxcomposite1 libxdamage1 libxfixes3 libxss1 libxtst6 libdrm2 libgbm1 libgtk-3-0 libglu1-mesa libsm6 libice6 libpng16-16'
    T64 = 'libasound2t64 libatk1.0-0t64 libatk-bridge2.0-0t64 libcups2t64'
    apt = subprocess.run(['bash', '-lc', 'apt-get update -qq && (apt-get install -y -qq ' + DEPS + ' ' + T64 + ' || apt-get install -y -qq ' + DEPS + ' libasound2 libatk1.0-0 libatk-bridge2.0-0 libcups2)'],
                         capture_output=True, text=True)
    print('deps install rc:', apt.returncode)
    if apt.returncode != 0:
        print('--- APT STDERR (tail) ---'); print((apt.stderr or '(empty)')[-2000:])
    miss = subprocess.run(['bash', '-lc', 'ldd "%s" | grep "not found" || true' % UNITY], capture_output=True, text=True).stdout.strip()
    print('loader-missing libs:', miss if miss else '(none)')

    def run(args, tag):
        r = subprocess.run(args, capture_output=True, text=True, timeout=600)
        out = (r.stdout or '') + (r.stderr or '')
        print('--- %s rc:' % tag, r.returncode)
        print(out[-3000:])
        return r.returncode

    rc1 = run([UCLIENT, '--show-seats', '--username', UNAME, '--password', UPASS], 'SHOW_SEATS')
    assert rc1 == 0, 'CREDENTIAL_PRECHECK_FAILED (143.002 = wrong password; 132.182 = blocked, wait 24h)'
    rc2 = run([UCLIENT, '--activate-all', '--include-personal', '--username', UNAME, '--password', UPASS], 'ACTIVATE')
    assert rc2 == 0, 'ACTIVATION FAILED'
    run([UCLIENT, '--showEntitlements'], 'ENTITLEMENTS')

rc3, bad, liclog = smoke()
print('EDITOR_SMOKE rc:', rc3, '| bad keywords:', bad or 'none')
assert rc3 == 0, 'GATE 2 FAIL - smoke exit %d\n%s' % (rc3, '\n'.join(liclog.splitlines()[-40:]))
print('GATE 2 PASS — license accepted headlessly')


In [ ]:
import subprocess, os, textwrap, time
# --- Cell 6: scripted project creation --- GATE 3 ---
UNITY = open('/content/unity_bin.txt').read().strip()
t0 = time.time()
r = subprocess.run(['bash', '-lc', '"%s" -batchmode -quit -nographics -createProject /content/probe -logFile /content/create.log; echo EXIT_CODE=$?' % UNITY],
                   capture_output=True, text=True, timeout=1800)
print('createProject:', [l for l in ((r.stdout or '') + (r.stderr or '')).splitlines() if l.startswith('EXIT_CODE=')], '| %.1f min' % ((time.time() - t0) / 60))
assert os.path.isdir('/content/probe/Assets'), 'project not created - log:\n' + open('/content/create.log', errors='ignore').read()[-3000:]
os.makedirs('/content/probe/Assets/Editor', exist_ok=True)
os.makedirs('/content/probe/Assets/Scenes', exist_ok=True)
open('/content/probe/Assets/Editor/BuildScript.cs', 'w').write(textwrap.dedent('''
    using System.IO; using System.Linq; using UnityEditor; using UnityEditor.SceneManagement; using UnityEngine;
    public static class BuildScript {
        [MenuItem("Build/Perform")]
        public static void PerformBuild() {
            var scenes = new string[] { "Assets/Scenes/Sample.unity" };
            if (!File.Exists("Assets/Scenes/Sample.unity")) {
                var scene = EditorSceneManager.NewScene(NewSceneSetup.DefaultGameObjects, NewSceneMode.Single);
                GameObject.CreatePrimitive(PrimitiveType.Cube);
                EditorSceneManager.SaveScene(scene, scenes[0]);
                Debug.Log("SCENE_CREATED_BY_SCRIPT");
            }
            EditorBuildSettings.scenes = scenes.Select(s => new EditorBuildSettingsScene(s, true)).ToArray();
            var report = BuildPipeline.BuildPlayer(scenes, "Builds/Linux/probe", BuildTarget.StandaloneLinux64, BuildOptions.None);
            if (report.summary.result != UnityEditor.Build.Reporting.BuildResult.Succeeded) throw new System.Exception("BUILD_FAILED: " + report.summary.result);
            Debug.Log("BUILD_OK " + report.summary.totalSize + " bytes");
        }
    }
'''))
print('GATE 3 PASS — project scaffolded; scripted scene+build in next cell')


In [ ]:
import subprocess, re, glob, os, time
# --- Cell 7: the build --- GATE 4 --- (-quit ALWAYS; timed; ALSA/XDG fix applied) ---
UNITY = open('/content/unity_bin.txt').read().strip()
t0 = time.time()
BUILD_CMD = ('export XDG_RUNTIME_DIR=/tmp/xdg; mkdir -p /tmp/xdg && chmod 700 /tmp/xdg; '
             'printf "pcm.!default {\n  type null\n}\nctl.!default {\n  type null\n}\n" > /etc/asound.conf; '
             'cd /content/probe && /usr/bin/time -v "%s" -batchmode -nographics -quit '
             '-projectPath /content/probe -executeMethod BuildScript.PerformBuild '
             '-logFile /content/build.log 2> /content/time.txt; echo EXIT_CODE=$?' % UNITY)
try:
    r = subprocess.run(['bash', '-lc', BUILD_CMD], capture_output=True, text=True, timeout=2700)
except Exception as e:
    print('RUNTIME_TIMEOUT_OR_ERROR:', repr(e))
    log = open('/content/build.log', errors='ignore').read() if os.path.exists('/content/build.log') else '(no log)'
    print('\n'.join(log.splitlines()[-50:]))
    raise
print('BASH_RETURN_CODE:', r.returncode)
peak = None
if os.path.exists('/content/time.txt'):
    for line in open('/content/time.txt', errors='ignore'):
        if line.startswith(('Elapsed (wall clock)', 'Maximum resident', 'Percent of CPU', 'Exit status')):
            print('MEASURE:', line.strip())
    m = re.search(r'Maximum resident set size.*?: (\d+)', open('/content/time.txt', errors='ignore').read())
    peak = int(m.group(1)) if m else None
log = open('/content/build.log', errors='ignore').read() if os.path.exists('/content/build.log') else '(no log)'
ok = 'BUILD_OK' in log
scene_created = 'SCENE_CREATED_BY_SCRIPT' in log
print('exit:', r.returncode, '| wall: %.1f min' % ((time.time() - t0) / 60), '| peak RAM: %.2f GB' % ((peak or 0) / 1048576.0),
      '| scene created by script:', scene_created, '| BUILD_OK:', ok)
print('output files:', sorted(glob.glob('/content/probe/Builds/Linux/*')))
if not ok:
    print('log tail:'); print('\n'.join(log.splitlines()[-50:]))
assert ok, 'BUILD FAILED'

# Player smoke — judged by LOG MARKERS, not exit code (headless Linux players may segv at exit; known pattern)
pr = subprocess.run(['bash', '-lc', 'export XDG_RUNTIME_DIR=/tmp/xdg; cd /content/probe/Builds/Linux && ./probe -batchmode -nographics -quit -logFile /tmp/probe_player.log; echo PLAYER_RC=$?'],
                    capture_output=True, text=True, timeout=900)
prc = [l for l in (pr.stdout or '').splitlines() if l.startswith('PLAYER_RC=')]
plog = open('/tmp/probe_player.log', errors='ignore').read() if os.path.exists('/tmp/probe_player.log') else ''
markers = [k for k in ['GfxDevice', 'Mono', 'Physics'] if k in plog]
print('player rc:', prc, '(segv-at-exit is a known headless pattern; verdict = markers)')
print('PLAYER_SMOKE:', 'PASS' if len(markers) >= 2 else 'FAIL', '| markers found:', markers)
print('GATE 4 PASS')


In [ ]:
import subprocess, os, time
# --- Cell 8: WebGL stretch gate (RAM ceiling canary) ---
t0 = time.time()
UNITY = open('/content/unity_bin.txt').read().strip()

def sh(cmd, tag, timeout=2400, tail=3000):
    r = subprocess.run(['bash', '-lc', cmd], capture_output=True, text=True, timeout=timeout)
    print('--- %s rc:%d' % (tag, r.returncode))
    out = (r.stdout or '') + (r.stderr or '')
    if out.strip(): print(out[-tail:])
    return r.returncode

sh('[ -d /content/ue/Editor/Data/PlaybackEngines/WebGLSupport ] && echo ALREADY_INSTALLED || aria2c -x8 -d /content -o WebGL.tar.xz "https://download.unity3d.com/download_unity/e35f0c77bd8e/LinuxEditorTargetInstaller/UnitySetup-WebGL-Support-for-Editor-6000.3.10f1.tar.xz"', 'DOWNLOAD_MODULE', timeout=900)
sh('[ -d /content/ue/Editor/Data/PlaybackEngines/WebGLSupport ] && echo ALREADY_EXTRACTED || (tar -xJf /content/WebGL.tar.xz -C /content/ue && echo EXTRACT_OK)', 'EXTRACT_MODULE', timeout=600)
sh('[ -d /content/ue/Editor/Data/PlaybackEngines/WebGLSupport ] && echo MODULE_PRESENT || echo MODULE_MISSING', 'MODULE_CHECK')
assert os.path.isdir('/content/ue/Editor/Data/PlaybackEngines/WebGLSupport'), 'WebGL module missing'

sh("sed -i 's/StandaloneLinux64/WebGL/g; s|Builds/Linux|Builds/WebGL|g' /content/probe/Assets/Editor/BuildScript.cs && grep -nE 'BuildTarget|Builds/' /content/probe/Assets/Editor/BuildScript.cs", 'RETARGET_SCRIPT')

BUILD_CMD = ('export XDG_RUNTIME_DIR=/tmp/xdg; '
             'cd /content/probe && /usr/bin/time -v "%s" -batchmode -nographics -quit '
             '-projectPath /content/probe -executeMethod BuildScript.PerformBuild '
             '-logFile /content/build_web.log 2> /content/time_web.txt' % UNITY)
rc = sh(BUILD_CMD, 'BUILD_WEBGL')
if rc != 0:
    tail_txt = open('/content/build_web.log', errors='ignore').read()[-3000:] if os.path.exists('/content/build_web.log') else '(no log)'
    print('BUILD_LOG_TAIL:', tail_txt)
    raise AssertionError('WEBGL_BUILD_FAILED')

print('BUILD_OK_LINE:', subprocess.run(['bash', '-lc', 'grep -a BUILD_OK /content/build_web.log'], capture_output=True, text=True).stdout.strip())
for line in open('/content/time_web.txt', errors='ignore'):
    if line.startswith(('Elapsed (wall clock)', 'Maximum resident', 'Percent of CPU', 'Exit status')):
        print('MEASURE:', line.strip())
sh('ls -la /content/probe/Builds/WebGL 2>/dev/null | head -20 || echo NO_ARTIFACTS', 'WEBGL_ARTIFACTS')
print('GATE_WEBGL: PASS | %.1f min' % ((time.time() - t0) / 60))


In [ ]:
import subprocess, os, json, glob
# --- Cell 9: final report — paste this whole output back to the agent ---
cfg = json.load(open('/content/probe_config.json'))
print('===== PROBE REPORT v3 (proven path) =====')
print('EDITOR:', cfg['VERSION'], '| REV:', cfg['REV'])
ubin = open('/content/unity_bin.txt').read().strip()
print('unity_bin:', ubin)
lic = os.path.expanduser('~/.config/unity3d/Unity/licenses/UnityEntitlementLicense.xml')
print('license xml:', 'PRESENT' if os.path.exists(lic) else 'MISSING', lic)
print('Builds/Linux:', sorted(glob.glob('/content/probe/Builds/Linux/*')))
print('Builds/WebGL:', sorted(glob.glob('/content/probe/Builds/WebGL/*')) or '(not run)')
print('DISK:', subprocess.run(['bash', '-lc', 'df -h /content | tail -1'], capture_output=True, text=True).stdout.strip())
print('Verdict lines: see GATE 1/2/3/4/GATE_WEBGL/PLAYER_SMOKE prints above.')
